In [36]:
from dotenv import load_dotenv
load_dotenv()

True

In [37]:
from anthropic import Anthropic
client = Anthropic()

In [38]:
def llm(prompt):
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.content[0].text

In [39]:
llm("Hey, what's up?")

"Hey! Not much, just here and ready to chat. What's going on with you?"

In [40]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [41]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [42]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 83},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [43]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [44]:
documents[230]

{'id': 'c180431de3',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 4: Analytics Engineering with dbt',
 'question': 'dbt + BigQuery: "Dataset was not found in location" / region mismatch errors',
 'answer': 'This is the single most common dbt+BigQuery problem in the course. The various "404 Not found: Dataset was not found in location X" errors all come down to the same root cause: a region mismatch between\n\n- the source dataset (e.g. `trips_data_all`),\n- the dbt-managed dev/prod dataset (e.g. `dbt_<initial><lastname>` or `prod`), and\n- the connection\'s configured location in dbt Cloud or `profiles.yml`.\n\nBigQuery cannot read and write across regions in a single query, and dbt creates new datasets in whatever location its connection is configured to use. If those don\'t match your source data\'s region, you\'ll see one of:\n\n```\n404 Not found: Dataset <project>:<schema> was not found in location <region>\nAccess Denied: ... or perhaps it does not exist in locatio

In [45]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [46]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [47]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [48]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [49]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [50]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [51]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [52]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [53]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [54]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [55]:
response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

In [56]:
response.content[0].text

'# Yes, you can join now!\n\nBased on the course information:\n\n✅ **You can start immediately** - The videos and GitHub materials are already available, and you can begin learning right away.\n\n✅ **You don\'t need to register first** - You can start learning and submitting homework without formal registration. Registration is just used to gauge interest.\n\n⚠️ **Important for certificates** - If you want to receive a certificate, you need to:\n- Submit your project **while submissions are still being accepted**\n- Complete the course as part of a "live" cohort (not self-paced)\n- Participate in peer-reviewing 3 capstone projects after submission\n\n## How to get started:\n\n1. Check the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n2. Review the [course GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n3. See the deadlines in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/)\n\nThen follow the typical workfl

In [57]:
response.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=539, output_tokens=281, output_tokens_details=None, server_tool_use=None, service_tier='standard')

In [58]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00166875

In [59]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

In [60]:
response.content[0].text

'# Yes, you can join now!\n\nBased on the course information, here\'s what you need to know:\n\n**You can start immediately** - the videos and materials are already available. You can begin learning whenever you want.\n\n**However, there\'s an important note about certificates:**\n- If you want to receive a certificate, you need to submit your project **while the course is still accepting submissions**\n- Certificates are only awarded to those who complete the course with the "live" cohort (which includes peer-reviewing 3 capstone projects)\n- Self-paced completion alone does not qualify for a certificate\n\n**To get started:**\n1. Check the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n2. Review the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n3. Visit the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n4. Check the deadlines in the [course management platform](https://courses.dat

In [64]:
# Combining the message history with the new user message: 

def llm(instructions, user_prompt, model="claude-haiku-4-5-20251001"):
    message_history = [
        {"role": "user", "content": user_prompt}
    ]

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=instructions,
        messages=message_history
    )

    return response.content[0].text

In [65]:
def rag(query, model="claude-haiku-4-5-20251001"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [66]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course now! 

However, there's an important thing to know: if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. 

Here's what you should do to get started:

1. Check out the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)
2. You can start whenever you want - the videos and materials are available
3. Follow the typical workflow: watch lesson videos → work through notebooks → read homework instructions → submit answers through the course platform before deadlines
4. Check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for deadline information

Good luck with the course!
